# Full Benchmark Run - Mixed-Model Dual-GPU Matrix

Kicks off the complete TQ-VLM-Bench matrix execution using the `Orchestrator`. Dual-GPU parallel execution, model-to-GPU pinning, and resume-from-checkpoint are enabled by default.

- 15 runtimes: baseline + 3 llama.cpp native KV quant + 8 TQ MSE + 3 TQ prod
- 11 benchmarks: 8 VLM + 3 Text
- Models: Thinking on GPU 0 / port 8080, Instruct on GPU 1 / port 8081
- Results stored at `../results/runs/*.json`
- Charts auto-generated at `../results/reports/<model_id>/` for mixed-model runs

In [ ]:
import os
import sys
from pathlib import Path

# Ensure llama.cpp shared libs are resolvable for the server subprocess
LCPP_LIB = Path('../../llama.cpp/build/bin').resolve()
os.environ['LD_LIBRARY_PATH'] = f"{LCPP_LIB}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Make tq_bench importable when running from notebooks/
sys.path.insert(0, str(Path('..').resolve()))

from tq_bench.config import load_runtimes, load_benchmarks, load_models
from tq_bench.orchestrator import Orchestrator, OrchestratorConfig
from tq_bench.reporters.export import export_csv, export_json
from tq_bench.reporters.summary import render_markdown_summary

In [ ]:
CONFIG_DIR = Path('../configs').resolve()
RESULTS_DIR = Path('../../results').resolve()

runtimes = load_runtimes(CONFIG_DIR / 'runtimes.yaml')
benchmarks = load_benchmarks(CONFIG_DIR / 'benchmarks.yaml')
models = load_models(CONFIG_DIR / 'models.yaml')

print(f'Runtimes:   {len(runtimes)}')
print(f'Benchmarks: {len(benchmarks)}')
print(f'Models:     {len(models)}')
print(f'Total cells per model: {len(runtimes) * len(benchmarks)}')
print(f'Total cells across all models: {len(runtimes) * len(benchmarks) * len(models)}')

In [ ]:
MODEL_IDS = ('qwen3_vl_2b_thinking', 'qwen3_vl_2b_instruct')
SERVER_BINARY = Path('../../llama.cpp/build/bin/llama-server').resolve()

cfg = OrchestratorConfig(
    checkpoint_path=RESULTS_DIR / 'runs' / 'checkpoint.json',
    results_dir=RESULTS_DIR,
    runtimes_yaml=CONFIG_DIR / 'runtimes.yaml',
    benchmarks_yaml=CONFIG_DIR / 'benchmarks.yaml',
    models_yaml=CONFIG_DIR / 'models.yaml',
    server_binary=SERVER_BINARY,
    model_ids=MODEL_IDS,
    num_gpus=2,
)
orchestrator = Orchestrator(cfg)
print(f'Model lanes: {MODEL_IDS}')

In [ ]:
# Dual-GPU parallel execution with resume support.
# Safe to re-run: completed cells are skipped via checkpoint.
orchestrator.run(parallel=True, resume=True)

In [ ]:
import json
import pandas as pd

runs_dir = RESULTS_DIR / 'runs'
rows = []
for p in sorted(runs_dir.glob('*.json')):
    if p.name == 'checkpoint.json':
        continue
    with open(p) as f:
        r = json.load(f)
    if 'runtime_id' not in r or 'benchmark_id' not in r:
        continue
    rows.append({
        'model_id':     r.get('model_id'),
        'runtime_id':   r.get('runtime_id'),
        'benchmark_id': r.get('benchmark_id'),
        'score':        r.get('score'),
        'n_samples':    r.get('n_samples'),
        'n_failed':     r.get('n_failed', 0),
        'wall_time_s':  r.get('wall_time_seconds'),
        'status':       r.get('status', 'ok'),
    })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} result cells')
print(f"Models in results: {sorted(df['model_id'].dropna().unique().tolist())}")
df.head(20)

In [ ]:
# Export consolidated CSV/JSON
reports_dir = RESULTS_DIR / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

records = df.to_dict(orient='records')
export_csv(records, reports_dir / 'full_run.csv')
export_json(records, reports_dir / 'full_run.json')
print(render_markdown_summary(records))

Charts are auto-generated at `../../results/reports/<model_id>/` by the orchestrator in mixed-model mode. See `03_analysis.ipynb` for interactive per-model analysis and custom plots.